IMPORTS

In [39]:
import numpy as np

INPUT

In [40]:
states = ["CP","IP"]
symbols = ["cola","ice_t","lem"]

PREPROCESSING

In [41]:
symbol_to_id = {s:i for i,s in enumerate(symbols)}
obs_sequence = ["lem","ice_t","cola"]
obs = [symbol_to_id[s] for s in obs_sequence]

COUNTS

In [42]:
number_of_states = len(states)
number_of_timesteps = len(obs)

INITIAL PROBS

In [43]:
pi = np.array([1.0,0.0])

TRANSITION PROBS

In [44]:
A = np.array([
    [0.7,0.3],
    [0.5,0.5]
])

EMISSION PROBS

In [45]:
B = np.array([
    [0.6,0.1,0.3],
    [0.1,0.7,0.2]
])

INITIALIZATION OF ALPHA

In [46]:
alpha = np.zeros((number_of_timesteps,number_of_states))

In [47]:
for state in range(number_of_states):
    alpha[0][state] = pi[state]*B[state][obs[0]]

COMPUTING ALPHA AT EVERY TIMESTEP

In [48]:
for timestep in range(1,number_of_timesteps):
    for state in range(number_of_states):
        alpha[timestep][state] = sum(
                                    alpha[timestep-1][hidden_state]*A[hidden_state][state] 
                                    for hidden_state in range(number_of_states)
                                    )*B[state][obs[timestep]]

INITIALIZATION OF BETA

In [49]:
beta = np.zeros((number_of_timesteps,number_of_states))

In [50]:
beta[number_of_timesteps-1] = 1

COMPUTING BETA AT EVERY TIMESTEP

In [51]:
for timestep in reversed(range(number_of_timesteps-1)):
    for state in range(number_of_states):
        beta[timestep][state] = sum(
                                    A[state][hidden_state]*B[hidden_state][obs[timestep+1]]*beta[timestep+1][hidden_state] 
                                    for hidden_state in range(number_of_states)
                                    )

COMPUTING STATES

In [52]:
best_states = np.argmax(gamma,axis=1)
decoded_sequence = [states[state] for state in best_states]

INITIALIZATION OF DELTA

In [53]:
delta = np.zeros((number_of_timesteps, number_of_states))
for state in range(number_of_states):
    delta[0][state] = pi[state] * B[state][obs[0]]

COMPUTING DELTA VALUES

In [54]:
for timestep in range(1, number_of_timesteps):
    for current_state in range(number_of_states):
        delta[timestep][current_state] = max(
            delta[timestep - 1][previous_state] * A[previous_state][current_state]
            for previous_state in range(number_of_states)
        ) * B[current_state][obs[timestep]]

DISPLAYING VALUES

In [55]:
print("Alpha:\n", alpha)
print("\nBeta:\n", beta)


# --------------------
# Final Probability using Forward
forward_prob = np.sum(alpha[number_of_timesteps-1])

# --------------------
# Final Probability using Backward
backward_prob = sum(
    pi[state] * B[state][obs[0]] * beta[0][state]
    for state in range(number_of_states)
)

# --------------------
# Output
print("\nForward Probability:", forward_prob)
print("Backward Probability:", backward_prob)

# MOST PROBABLE STATE AT EACH TIMESTEP USING DELTA
best_states = np.argmax(delta, axis=1)

decoded_sequence = [states[state] for state in best_states]

# PROBABILITY OF BEST PATH
best_path_prob = np.max(delta[number_of_timesteps - 1])

print("Best sequence:", decoded_sequence)
print("Best path probability:", best_path_prob)

Alpha:
 [[0.3     0.     ]
 [0.021   0.063  ]
 [0.02772 0.00378]]

Beta:
 [[0.105 0.145]
 [0.45  0.35 ]
 [1.    1.   ]]

Forward Probability: 0.0315
Backward Probability: 0.03149999999999999
Best sequence: ['CP', 'IP', 'CP']
Best path probability: 0.0189
